# YOLO Colab 訓練筆記本

這份筆記本由 `tools/yolo_trainer_colab/train_colab.py` 轉換而來，目標是在 Google Colab 直接完成 Drive 掛載、套件安裝、資料集解壓、`data.yaml` 路徑修正、模型訓練與輸出打包。

建議先把資料集 zip 與模型權重放到 Google Drive。Notebook 預設採用 `fast` 模式：從 Drive 讀取輸入，但解壓與訓練過程放在 Colab 本機 `/content`，最後只把訓練結果 zip 寫回 Drive。

- `/content/drive/MyDrive/YOLOTools/trainingData/dataset.zip`
- `/content/drive/MyDrive/YOLOTools/models/yolov26x.pt`
- 輸出會寫到 `/content/drive/MyDrive/YOLOTools/output_zips/`


## 1. 安裝 Colab 需要的 Python 套件

Colab 通常已內建 PyTorch 與 CUDA 對應版本，因此這裡不強制重裝 `torch`，避免破壞 Colab GPU runtime。若你的 runtime 缺少 PyTorch，請先依 Colab 官方建議安裝對應版本。

In [ ]:
%pip install -q "ultralytics==8.4.14" "PyYAML==6.0.3" "pillow>=10.0.0" "requests>=2.32.0" "opencv-python-headless>=4.10.0"


## 2. 連接 Google Drive

執行後 Colab 會要求授權。授權完成後，雲端硬碟會掛載在 `/content/drive`。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. 檢查 GPU 與基本環境

In [ ]:
from pathlib import Path
import platform

import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f"GPU {index}: {torch.cuda.get_device_name(index)}")


## 4. 設定訓練參數

請依你的 Drive 路徑調整 `DATASET_ZIP` 與 `MODEL`。`MODEL` 可以是 Drive 裡的 `.pt` 檔，也可以是 Ultralytics 支援的模型名稱。

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "YOLOTools"

# fast: 解壓與訓練過程放在 /content，速度較快且較不容易被 Drive I/O 中斷。
# persistent: 解壓與訓練過程放在 Google Drive，可跨 runtime 保留，但大量小檔 I/O 較慢。
STORAGE_MODE = "fast"
LOCAL_CACHE_DIR = Path("/content/yolo_colab_cache")
LOCAL_WORK_DIR = Path("/content/yolo_colab_workdir")
DRIVE_WORK_DIR = PROJECT_DIR / "colab_workdir"

CONFIG = {
    "task": "detect",
    "dataset_zip": str(PROJECT_DIR / "trainingData" / "TaichungFireDataset-20260507.zip"),
    "storage_mode": STORAGE_MODE,
    "local_cache_dir": str(LOCAL_CACHE_DIR),
    "work_dir": str(LOCAL_WORK_DIR if STORAGE_MODE == "fast" else DRIVE_WORK_DIR),
    "out_zip_dir": str(PROJECT_DIR / "output_zips"),
    "model": "yolo26x.pt",
    "model_dir": str(PROJECT_DIR / "models"),
    "epochs": 120,
    "imgsz": 640,
    "batch": 12,
    "device": "",
    "resume": False,
    "skip_unlabeled": False,
    "delete_temp": False,
}

CONFIG


## 5. 載入訓練工具函式

In [ ]:
from __future__ import annotations

import csv
import hashlib
import shutil
import zipfile
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Callable

import yaml


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_EXTENSION = ".txt"
MANIFEST_FILE_NAME = "_manifest.yaml"
PATCHED_DIR_NAME = "_patched"
IGNORED_MANIFEST_PARTS = {MANIFEST_FILE_NAME, PATCHED_DIR_NAME, "__MACOSX", ".ipynb_checkpoints"}


@dataclass(frozen=True)
class YamlMappingModel:
    """YAML mapping serializer for dataset and manifest files.

    Args:
        data: YAML mapping content.
    """

    data: dict[str, object]

    @classmethod
    def from_file(cls, path: Path) -> "YamlMappingModel":
        """Load a YAML mapping from disk.

        Args:
            path: YAML file path.

        Returns:
            Parsed YAML mapping model.

        Raises:
            ValueError: If the YAML document is not a mapping.
        """
        obj = yaml.safe_load(path.read_text(encoding="utf-8"))
        if not isinstance(obj, dict):
            raise ValueError(f"YAML document must be a mapping. path={path}")
        return cls(data=dict(obj))

    def save(self, path: Path) -> None:
        """Write the YAML mapping to disk.

        Args:
            path: Destination file path.
        """
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(
            yaml.safe_dump(self.data, sort_keys=False, allow_unicode=True),
            encoding="utf-8",
        )


@dataclass(frozen=True)
class TrainConfig:
    """Training configuration for Colab YOLO runs.

    Args:
        task: Ultralytics task, such as detect, segment, classify, pose, or obb.
        dataset_zip: Dataset zip path on Google Drive.
        storage_mode: Storage mode. Use fast for local Colab workdir or persistent for Drive workdir.
        local_cache_dir: Local cache directory used to stage Drive zip files in fast mode.
        work_dir: Working directory for extracted data and Ultralytics runs.
        out_zip_dir: Directory where zipped training output will be saved.
        model: Model weight path or Ultralytics model name.
        model_dir: Directory used to resolve model weights from Drive.
        epochs: Training epoch count.
        imgsz: Training image size.
        batch: Training batch size.
        device: Device hint. Empty string auto-selects CUDA when available.
        resume: Whether to resume a previous Ultralytics run.
        skip_unlabeled: Whether to remove unlabeled train and validation images.
        delete_temp: Whether to remove extracted dataset files after training.
    """

    task: str
    dataset_zip: str
    storage_mode: str
    local_cache_dir: str
    work_dir: str
    out_zip_dir: str
    model: str
    model_dir: str
    epochs: int
    imgsz: int
    batch: int
    device: str
    resume: bool
    skip_unlabeled: bool
    delete_temp: bool


def now_str() -> str:
    """Return a timestamp string for notebook logs.

    Returns:
        Current local time formatted as a readable timestamp.
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def log(message: str) -> None:
    """Print a log message immediately.

    Args:
        message: Text to print.
    """
    print(message, end="", flush=True)


def load_config(data: dict[str, object]) -> TrainConfig:
    """Deserialize a notebook configuration mapping.

    Args:
        data: Raw configuration mapping.

    Returns:
        Validated training configuration.

    Raises:
        ValueError: If numeric parameters are invalid.
    """
    epochs = int(data.get("epochs", 50))
    imgsz = int(data.get("imgsz", 640))
    batch = int(data.get("batch", 16))
    if epochs <= 0:
        raise ValueError(f"epochs must be positive. epochs={epochs}")
    if imgsz <= 0:
        raise ValueError(f"imgsz must be positive. imgsz={imgsz}")
    if batch == 0:
        raise ValueError("batch must not be zero")
    storage_mode = str(data.get("storage_mode", "fast")).strip().lower()
    if storage_mode not in ("fast", "persistent"):
        raise ValueError(f"storage_mode must be fast or persistent. storage_mode={storage_mode}")
    return TrainConfig(
        task=str(data.get("task", "detect")),
        dataset_zip=str(data.get("dataset_zip", "")),
        storage_mode=storage_mode,
        local_cache_dir=str(data.get("local_cache_dir", "/content/yolo_colab_cache")),
        work_dir=str(data.get("work_dir", "/content/yolo_workdir")),
        out_zip_dir=str(data.get("out_zip_dir", "/content/yolo_output_zips")),
        model=str(data.get("model", "yolov8n.pt")),
        model_dir=str(data.get("model_dir", "")),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=str(data.get("device", "")),
        resume=bool(data.get("resume", False)),
        skip_unlabeled=bool(data.get("skip_unlabeled", False)),
        delete_temp=bool(data.get("delete_temp", False)),
    )


def validate_zip_file(zip_path: Path) -> None:
    """Validate that a dataset zip can be opened and read.

    Args:
        zip_path: Dataset zip path.

    Raises:
        zipfile.BadZipFile: If the zip file is corrupt or unreadable.
    """
    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_member = archive.testzip()
    if bad_member:
        raise zipfile.BadZipFile(f"zip 內檔案損壞：{bad_member}")


def prepare_dataset_zip(dataset_zip: Path, local_cache_dir: Path, storage_mode: str, log_cb: Callable[[str], None]) -> Path:
    """Prepare the dataset zip for extraction.

    Args:
        dataset_zip: Original dataset zip path.
        local_cache_dir: Local cache directory for fast mode.
        storage_mode: Configured storage mode.
        log_cb: Logging callback.

    Returns:
        Dataset zip path to use for extraction.

    Raises:
        OSError: If copying from Drive to local cache fails.
        zipfile.BadZipFile: If the local zip validation fails.
    """
    if storage_mode != "fast":
        validate_zip_file(dataset_zip)
        return dataset_zip
    local_cache_dir.mkdir(parents=True, exist_ok=True)
    local_zip = local_cache_dir / dataset_zip.name
    source_size = dataset_zip.stat().st_size
    if local_zip.exists() and local_zip.stat().st_size == source_size:
        log_cb(f"[{now_str()}] 使用既有本機 dataset zip 快取：{local_zip}\n")
    else:
        log_cb(f"[{now_str()}] 複製 dataset zip 到本機快取：{local_zip}\n")
        shutil.copy2(dataset_zip, local_zip)
    validate_zip_file(local_zip)
    return local_zip


def extract_zip(zip_path: Path, out_dir: Path, name_prefix: str) -> Path:
    """Extract a dataset zip into a named work directory.

    Args:
        zip_path: Dataset zip path.
        out_dir: Parent directory for extraction.
        name_prefix: Directory name used for the extracted dataset.

    Returns:
        Extracted dataset root path.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    root = out_dir / name_prefix
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(root)
    return resolve_extracted_dataset_root(root)


def resolve_extracted_dataset_root(container_root: Path) -> Path:
    """Resolve the actual dataset root inside an extraction container.

    Args:
        container_root: Directory named from the dataset zip stem.

    Returns:
        Actual dataset root. If the zip contains one top-level dataset folder,
        that inner folder is returned.
    """
    if not container_root.exists():
        return container_root
    child_dirs = [
        path for path in container_root.iterdir()
        if path.is_dir() and path.name not in IGNORED_MANIFEST_PARTS
    ]
    child_files = [
        path for path in container_root.iterdir()
        if path.is_file() and path.name not in IGNORED_MANIFEST_PARTS
    ]
    if len(child_dirs) == 1 and not child_files:
        return child_dirs[0]
    return container_root


def _is_manifest_ignored(path: Path, root: Path) -> bool:
    try:
        relative = path.relative_to(root)
    except ValueError:
        relative = path
    return any(part in IGNORED_MANIFEST_PARTS for part in relative.parts)


def _iter_files_for_hash(root: Path) -> list[Path]:
    return [path for path in root.rglob("*") if path.is_file() and not _is_manifest_ignored(path, root)]


def _hash_file_list(root: Path, paths: list[Path]) -> str:
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: str(item.relative_to(root)).lower()):
        try:
            relative = str(path.relative_to(root)).replace("\\", "/")
            digest.update(relative.encode("utf-8", errors="ignore"))
            digest.update(str(path.stat().st_size).encode("utf-8"))
        except OSError:
            continue
    return digest.hexdigest()


def _legacy_hash_file_list(paths: list[Path]) -> str:
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: str(item).lower()):
        try:
            digest.update(str(path).replace("\\", "/").encode("utf-8", errors="ignore"))
            digest.update(str(path.stat().st_size).encode("utf-8"))
        except OSError:
            continue
    return digest.hexdigest()


def compute_dataset_manifest(root: Path) -> dict[str, object]:
    """Compute a lightweight dataset manifest.

    Args:
        root: Extracted dataset root.

    Returns:
        Manifest containing file count and content-size hash.
    """
    files = _iter_files_for_hash(root)
    return {"file_count": len(files), "hash": _hash_file_list(root, files)}


def manifest_matches(root: Path, expected: dict[str, object]) -> bool:
    """Check whether the current dataset matches a stored manifest.

    Args:
        root: Extracted dataset root.
        expected: Previously stored manifest.

    Returns:
        True when file count and hash match.
    """
    current = compute_dataset_manifest(root)
    if current.get("file_count") == expected.get("file_count") and current.get("hash") == expected.get("hash"):
        return True
    files = _iter_files_for_hash(root)
    legacy = {"file_count": len(files), "hash": _legacy_hash_file_list(files)}
    return legacy.get("file_count") == expected.get("file_count") and legacy.get("hash") == expected.get("hash")


def write_manifest(path: Path, data: dict[str, object]) -> None:
    """Write a dataset manifest YAML file.

    Args:
        path: Manifest destination path.
        data: Manifest mapping.
    """
    YamlMappingModel(data=dict(data)).save(path)


def read_manifest(path: Path) -> dict[str, object] | None:
    """Read a dataset manifest if it exists.

    Args:
        path: Manifest path.

    Returns:
        Manifest mapping, or None when unavailable.
    """
    if not path.exists():
        return None
    try:
        return dict(YamlMappingModel.from_file(path).data)
    except (OSError, ValueError, yaml.YAMLError):
        return None


def find_data_yaml(root: Path) -> Path | None:
    """Find a YOLO data.yaml file under the extracted dataset.

    Args:
        root: Extracted dataset root.

    Returns:
        Path to data.yaml or data.yml, or None if not found.
    """
    for path in (root / "data.yaml", root / "data.yml"):
        if path.exists():
            return path
    for path in list(root.rglob("data.yaml")) + list(root.rglob("data.yml")):
        return path
    return None


def _resolve_path(base_dir: Path, value: str) -> Path:
    candidate = Path(value)
    return candidate if candidate.is_absolute() else (base_dir / candidate).resolve()


def _iter_images(image_dir: Path) -> list[Path]:
    if not image_dir.exists():
        return []
    return [path for path in image_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]


def filter_unlabeled_yolo_dataset(data_yaml_path: Path, log_cb: Callable[[str], None], apply_to: tuple[str, ...] = ("train", "val")) -> dict[str, int]:
    """Remove images without matching YOLO label files.

    Args:
        data_yaml_path: Patched data.yaml path.
        log_cb: Logging callback.
        apply_to: Dataset splits to filter.

    Returns:
        Count of checked and removed images.
    """
    base_dir = data_yaml_path.parent
    data = dict(YamlMappingModel.from_file(data_yaml_path).data)
    checked_total = 0
    removed_total = 0
    for split in apply_to:
        if split not in data:
            continue
        split_path = _resolve_path(base_dir, str(data[split]))
        image_dir = split_path
        images = _iter_images(image_dir)
        if not images:
            candidate = split_path / "images" / split
            images = _iter_images(candidate)
            if images:
                image_dir = candidate
        parts = list(image_dir.parts)
        if "images" in parts:
            index = parts.index("images")
            labels_dir = Path(*parts[:index], "labels", *parts[index + 1:])
        else:
            labels_dir = image_dir.parent / "labels"
        checked = 0
        removed = 0
        for image_path in images:
            checked += 1
            checked_total += 1
            try:
                relative = image_path.relative_to(image_dir)
            except ValueError:
                relative = Path(image_path.name)
            label_path = (labels_dir / relative).with_suffix(LABEL_EXTENSION)
            if not label_path.exists() or label_path.stat().st_size == 0:
                image_path.unlink(missing_ok=True)
                removed += 1
                removed_total += 1
        log_cb(f"  - split={split}: checked={checked}, removed_unlabeled={removed}\n")
    return {"checked": checked_total, "removed": removed_total}


def rewrite_data_yaml_to_extracted_root(orig_yaml: Path, extracted_root: Path, log_cb: Callable[[str], None]) -> Path:
    """Patch data.yaml so Ultralytics reads from the extracted Drive dataset.

    Args:
        orig_yaml: Original YOLO data.yaml path.
        extracted_root: Extracted dataset root.
        log_cb: Logging callback.

    Returns:
        Path to patched data.yaml.
    """
    data = dict(YamlMappingModel.from_file(orig_yaml).data)
    patched_yaml = extracted_root / "_patched" / "data.yaml"

    def normalize_relative(value: str) -> str:
        return value.replace("\\", "/").lstrip("./")

    def best_relative_from_absolute(value: str) -> str:
        normalized = value.replace("\\", "/")
        markers = (
            "/images/train", "/images/val", "/images/test",
            "/images/Train", "/images/Val", "/images/Test",
            "/labels/train", "/labels/val", "/labels/test",
        )
        for marker in markers:
            index = normalized.lower().find(marker.lower())
            if index != -1:
                return normalize_relative(normalized[index + 1:])
        parts = [part for part in normalized.split("/") if part]
        if len(parts) >= 2:
            return normalize_relative("/".join(parts[-2:]))
        return normalize_relative(parts[-1]) if parts else "images/train"

    data["path"] = str(extracted_root)
    for split in ("train", "val", "test"):
        if split not in data:
            continue
        value = str(data[split])
        if Path(value).is_absolute():
            relative = best_relative_from_absolute(value)
            data[split] = relative
            log_cb(f"  - rewrite {split}: ABS -> REL  {value}  =>  {relative}\n")
        else:
            data[split] = normalize_relative(value)
            log_cb(f"  - normalize {split}: {value} => {data[split]}\n")
    YamlMappingModel(data=dict(data)).save(patched_yaml)
    log_cb(f"[patched] data.yaml => {patched_yaml}\n")
    return patched_yaml


def parse_results_csv(run_dir: Path) -> dict[str, Any]:
    """Parse the final metrics from an Ultralytics results.csv file.

    Args:
        run_dir: Ultralytics run directory.

    Returns:
        Metrics payload containing parsed metrics and raw rows.
    """
    csv_path = run_dir / "results.csv"
    if not csv_path.exists():
        return {}
    with csv_path.open("r", encoding="utf-8") as file_obj:
        rows = list(csv.DictReader(file_obj))
    if not rows:
        return {}
    last = rows[-1]
    previous = rows[-2] if len(rows) > 1 else None

    def get_float(keys: list[str]) -> float | None:
        for key in keys:
            if key in last and last[key] not in (None, ""):
                try:
                    return float(last[key])
                except ValueError:
                    continue
        return None

    precision = get_float(["metrics/precision(B)", "metrics/precision", "precision"])
    recall = get_float(["metrics/recall(B)", "metrics/recall", "recall"])
    map50 = get_float(["metrics/mAP50(B)", "metrics/mAP50", "mAP50"])
    map5095 = get_float(["metrics/mAP50-95(B)", "metrics/mAP50-95", "mAP50-95", "mAP5095"])
    f1_score = None
    if precision is not None and recall is not None and precision + recall > 0:
        f1_score = 2 * precision * recall / (precision + recall)
    return {
        "metrics": {"precision": precision, "recall": recall, "f1": f1_score, "mAP50": map50, "mAP50-95": map5095},
        "last_row": last,
        "prev_row": previous,
    }


def read_last_epoch_row(run_dir: Path) -> dict[str, Any] | None:
    """Read the last row from an Ultralytics results.csv file.

    Args:
        run_dir: Ultralytics run directory.

    Returns:
        Last CSV row, or None when unavailable.
    """
    csv_path = run_dir / "results.csv"
    if not csv_path.exists():
        return None
    with csv_path.open("r", encoding="utf-8") as file_obj:
        rows = list(csv.DictReader(file_obj))
    return rows[-1] if rows else None


def zip_folder(src_dir: Path, zip_path: Path, log_cb: Callable[[str], None]) -> None:
    """Zip a directory recursively.

    Args:
        src_dir: Source directory.
        zip_path: Destination zip path.
        log_cb: Logging callback.
    """
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    log_cb(f"打包輸出 zip: {zip_path}\n")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in src_dir.rglob("*"):
            if path.is_file():
                archive.write(path, arcname=str(path.relative_to(src_dir)))


def remove_dir_safe(path: Path, log_cb: Callable[[str], None]) -> None:
    """Remove a temporary directory with contextual logging.

    Args:
        path: Directory to remove.
        log_cb: Logging callback.
    """
    try:
        if path.exists():
            shutil.rmtree(path, ignore_errors=True)
            log_cb(f"已刪除暫存資料夾：{path}\n")
    except OSError as exc:
        log_cb(f"刪除暫存資料夾失敗：{exc}\n")


def normalize_device_arg(device_hint: str) -> str:
    """Normalize the device argument for Colab.

    Args:
        device_hint: User-provided device string.

    Returns:
        A valid Ultralytics device argument.
    """
    import torch

    device = device_hint.strip()
    if device.lower() == "cpu":
        return "cpu"
    if not torch.cuda.is_available():
        return "cpu"
    device_count = torch.cuda.device_count()
    if not device:
        return "cuda:0"
    text = device.split(":", 1)[1] if device.startswith("cuda:") else device
    requested = [int(item.strip()) for item in text.split(",") if item.strip().isdigit()]
    valid = []
    for index in requested:
        if 0 <= index < device_count and index not in valid:
            valid.append(index)
    if not valid:
        return "cuda:0"
    return "cuda:" + ",".join(str(index) for index in valid)


def run_training(config_data: dict[str, object]) -> dict[str, object]:
    """Run the full Colab YOLO training workflow.

    Args:
        config_data: Notebook configuration mapping.

    Returns:
        Summary payload with paths and metrics.

    Raises:
        FileNotFoundError: If dataset zip or data.yaml is missing.
        RuntimeError: If training completes without a run directory.
    """
    from ultralytics import YOLO

    cfg = load_config(config_data)
    source_dataset_zip = Path(cfg.dataset_zip).expanduser().resolve()
    if not source_dataset_zip.exists():
        raise FileNotFoundError(f"dataset.zip 不存在：{source_dataset_zip}")
    work_dir = Path(cfg.work_dir).expanduser().resolve()
    out_zip_dir = Path(cfg.out_zip_dir).expanduser().resolve()
    local_cache_dir = Path(cfg.local_cache_dir).expanduser().resolve()
    model_dir = Path(cfg.model_dir).expanduser().resolve() if cfg.model_dir else None
    work_dir.mkdir(parents=True, exist_ok=True)
    out_zip_dir.mkdir(parents=True, exist_ok=True)
    if model_dir:
        model_dir.mkdir(parents=True, exist_ok=True)

    log(f"[{now_str()}] ===== 開始訓練 =====\n")
    log(f"[{now_str()}] task={cfg.task}\n")
    log(f"[{now_str()}] storage_mode={cfg.storage_mode}\n")
    log(f"[{now_str()}] dataset.zip={source_dataset_zip}\n")
    log(f"[{now_str()}] work_dir={work_dir}\n")
    log(f"[{now_str()}] out_zip_dir={out_zip_dir}\n")
    log(f"[{now_str()}] model={cfg.model}\n")
    log(f"[{now_str()}] epochs={cfg.epochs}, imgsz={cfg.imgsz}, batch={cfg.batch}, device={cfg.device}, resume={cfg.resume}\n\n")

    dataset_zip = prepare_dataset_zip(source_dataset_zip, local_cache_dir, cfg.storage_mode, log)
    log(f"[{now_str()}] active dataset.zip={dataset_zip}\n")

    name_prefix = f"ds_{source_dataset_zip.stem}"
    container_root = work_dir / name_prefix
    extracted_root = resolve_extracted_dataset_root(container_root)
    manifest_path = extracted_root / MANIFEST_FILE_NAME
    manifest_candidates = [manifest_path]
    legacy_manifest_path = container_root / MANIFEST_FILE_NAME
    if legacy_manifest_path != manifest_path:
        manifest_candidates.append(legacy_manifest_path)
    reuse = False
    if container_root.exists():
        log(f"[{now_str()}] 檢查資料集（可否重用）...\n")
        for candidate in manifest_candidates:
            expected = read_manifest(candidate)
            if expected and manifest_matches(extracted_root, expected):
                reuse = True
                if candidate != manifest_path:
                    write_manifest(manifest_path, compute_dataset_manifest(extracted_root))
                log(f"[{now_str()}] 使用既有解壓資料：{extracted_root}\n")
                break
    if not reuse:
        log(f"[{now_str()}] 解壓縮資料集...\n")
        extracted_root = extract_zip(dataset_zip, work_dir, name_prefix)
        manifest_path = extracted_root / MANIFEST_FILE_NAME
        log(f"[{now_str()}] 解壓縮完成：{extracted_root}\n")
        write_manifest(manifest_path, compute_dataset_manifest(extracted_root))

    data_yaml = find_data_yaml(extracted_root)
    if not data_yaml:
        raise FileNotFoundError(f"找不到 data.yaml：{extracted_root}")
    log(f"[{now_str()}] 找到原始 data.yaml：{data_yaml}\n")
    log(f"[{now_str()}] 修正 data.yaml 路徑（指向解壓資料夾）...\n")
    patched_yaml = rewrite_data_yaml_to_extracted_root(data_yaml, extracted_root, log)
    log(f"[{now_str()}] 使用 patched data.yaml：{patched_yaml}\n")

    if cfg.skip_unlabeled and cfg.task in ("detect", "segment", "pose", "obb"):
        log(f"[{now_str()}] 清理未標記圖片（skip unlabeled）...\n")
        stats = filter_unlabeled_yolo_dataset(patched_yaml, log, apply_to=("train", "val"))
        log(f"[{now_str()}] 清理完成：checked={stats['checked']}, removed={stats['removed']}\n")

    model_path = cfg.model
    drive_model_path = None
    if model_dir and not Path(cfg.model).expanduser().exists():
        drive_model_path = model_dir / Path(cfg.model).name
        if drive_model_path.exists():
            model_path = str(drive_model_path)
    try:
        model = YOLO(model_path)
    except Exception as exc:
        hint = f" 請確認模型權重是否存在於：{drive_model_path}" if drive_model_path else ""
        raise FileNotFoundError(
            f"模型載入失敗：{model_path}.{hint} 或將 CONFIG['model'] 改成 Ultralytics 支援的模型名稱。"
        ) from exc
    log(f"[{now_str()}] 模型：{model_path}\n")

    run_name = f"{cfg.task}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    train_kwargs: dict[str, object] = {
        "data": str(patched_yaml) if cfg.task != "classify" else str(extracted_root),
        "epochs": cfg.epochs,
        "imgsz": cfg.imgsz,
        "batch": cfg.batch,
        "project": str(work_dir / "runs"),
        "name": run_name,
        "exist_ok": False,
        "verbose": True,
    }
    normalized_device = normalize_device_arg(cfg.device)
    if normalized_device:
        train_kwargs["device"] = normalized_device
    if cfg.resume:
        train_kwargs["resume"] = True
    results = model.train(**train_kwargs)

    run_dir = None
    try:
        run_dir = Path(results.save_dir)
    except AttributeError:
        trainer = getattr(model, "trainer", None)
        if trainer is not None and getattr(trainer, "save_dir", None):
            run_dir = Path(trainer.save_dir)
    if not run_dir or not run_dir.exists():
        candidates = sorted((work_dir / "runs").rglob("results.csv"), key=lambda path: path.stat().st_mtime, reverse=True)
        if candidates:
            run_dir = candidates[0].parent
    if not run_dir or not run_dir.exists():
        raise RuntimeError("訓練完成但找不到 run_dir（無法整理輸出）")

    log(f"[{now_str()}] run_dir：{run_dir}\n")
    metrics_payload = parse_results_csv(run_dir)
    log(f"[{now_str()}] 上一輪指標：{read_last_epoch_row(run_dir) or metrics_payload.get('last_row')}\n")
    out_zip_path = out_zip_dir / f"{run_dir.name}.zip"
    zip_folder(run_dir, out_zip_path, log)
    if cfg.delete_temp:
        remove_dir_safe(extracted_root, log)
    log(f"\n[{now_str()}] ===== 完成 =====\n")
    return {
        "run_dir": str(run_dir),
        "out_zip": str(out_zip_path),
        "data_yaml": str(patched_yaml),
        "metrics": metrics_payload.get("metrics", {}) if metrics_payload else {},
        "last_row": metrics_payload.get("last_row") if metrics_payload else None,
    }


## 6. 檢查 Drive 路徑

這個 cell 只檢查資料集 zip 是否存在，不會開始訓練。

In [ ]:
dataset_zip = Path(CONFIG["dataset_zip"])
model_dir = Path(CONFIG["model_dir"])
work_dir = Path(CONFIG["work_dir"])
out_zip_dir = Path(CONFIG["out_zip_dir"])
local_cache_dir = Path(CONFIG["local_cache_dir"])

print(f"Storage mode: {CONFIG['storage_mode']}")
print(f"Dataset zip: {dataset_zip}")
print(f"Dataset exists: {dataset_zip.exists()}")
print(f"Local cache directory: {local_cache_dir}")
print(f"Work directory: {work_dir}")
print(f"Output zip directory: {out_zip_dir}")
print(f"Model directory: {model_dir}")
print(f"Model directory exists: {model_dir.exists()}")

if not dataset_zip.exists():
    raise FileNotFoundError(f"請先把資料集 zip 放到指定路徑：{dataset_zip}")


## 7. 開始訓練並打包輸出

In [ ]:
summary = run_training(CONFIG)
summary
